In [59]:
from agent_framework import (
    Executor,
    FunctionExecutor,
    WorkflowContext,
    handler,
    executor,
    WorkflowBuilder,
)
import inspect
from typing import get_origin, Never

## Signature / Outputs Guide

`ctx: WorkflowContext[str]`: used with send_message to send a [str] message to the next executor

`ctx: WorkflowContext[Never, str]`: used with ctx.yield_output to send a [str] workflow output to the caller

`ctx: WorkflowContext[str, str]`: send_message[str] and yield_output[str]


Explicitly define type parameters:

`@handler(input=str | int, output=str, workflow_output=bool)`: 

Implicitly define type parameters (will be inferred through introspection)

```
@handler()
async def my_method(self, input: str | int, ctx: WorkflowContext[str, bool])
``` 



In [60]:
class UpperCase(Executor):
    def __init__(self, id: str = "upper_case_executor") -> None:
        super().__init__(id=id)
    @handler
    async def to_upper_case(self, text: str, ctx: WorkflowContext[str]) -> None:
        """Convert the input to uppercase and forward it to the next node."""
        await ctx.send_message(text.upper())

    # Note: this handle will always be called for a str type because handlers are registered in reverse order, so the most recently added handler that matches the input type will be used.
    @handler
    async def to_lower_case(self, text: str | int, ctx: WorkflowContext[str]) -> None:
        """Convert the input to lowercase and forward it to the next node."""
        await ctx.send_message(str(text)[:3].lower())

    @handler
    async def to_double_integer(self, number: int, ctx: WorkflowContext[int]) -> None:
        """Double the input integer and forward it to the next node."""
        await ctx.send_message(number * 2)

In [ ]:
# Note: @executor expects your function to accept 1 or 2 params (message) or (message, context) it
# blindly treats params[0] as the message regardless of name or annotation. This means if your
# first parameter is context, it will assume that is actually your message. This is a silent failure that should be avoided.
@executor()
async def reverse_text(text: str, ctx: WorkflowContext[str, int]) -> None:
    """Reverse the input text and forward it to the next node."""
    await ctx.send_message(text[::-1])

# Note: the explicit types are not checked by the framework. If you provide incorrect types, it may lead to runtime errors or unexpected behavior.
@executor(input=str, output=str, workflow_output=float)
async def reverse_text_typed(text, ctx: WorkflowContext[Never, int]) -> None:
    """Reverse the input text and forward it to the next node."""
    await ctx.send_message(text[::-1])

In [62]:
upper_case_executor = UpperCase()
upper_case_executor.type

'UpperCase'

In [63]:
upper_case_executor.to_json()

'{"id": "upper_case_executor", "type": "UpperCase"}'

In [64]:
reverse_text.to_json()

'{"id": "reverse_text", "type": "FunctionExecutor"}'

In [65]:
reverse_text.input_types

[str]

In [66]:
reverse_text.output_types

[str]

In [67]:
reverse_text.type

'FunctionExecutor'

In [68]:
from typing import TypedDict


class HandlerSpecInfo(TypedDict):
    handler_name: str
    input_type: list[type]
    output_types: list[type]
    workflow_output_types: list[type]
    doc_string: str | None
    input_parameter_names: list[str]

In [69]:
def get_handler_doc_string(executor: Executor, input_type: type) -> str | None:
    return executor._handlers.get(input_type).__doc__

# Note: Executor prevents handlers from having more than one message_type
def get_handler_parameter_names(executor: Executor, input_type: type) -> list[str]:
    print(input_type)
    handler = executor._handlers.get(input_type)
    if handler is None:
        raise ValueError(f"No handler found for input type {input_type}")
    parameters = inspect.signature(handler).parameters
    parameter_names = [parameters[p].name for p in parameters if not get_origin(parameters[p].annotation) is WorkflowContext] # remove the Workflow context as it is not user provided input
    return parameter_names

def get_function_executor_metadata(executor: FunctionExecutor) -> list[HandlerSpecInfo]:
    if not isinstance(executor, FunctionExecutor):
        raise ValueError("Input must be an instance of Executor")
    handler_info: list[HandlerSpecInfo] = []
    # Note: Use executor._original_func to access the original function
    for handler_spec in executor._handler_specs:
        name = handler_spec["name"]
        input_type = handler_spec["message_type"]
        output_types = handler_spec["output_types"]
        workflow_output_types = handler_spec["workflow_output_types"]

        handler_info.append({
            "handler_name": name,
            "input_type": input_type,
            "output_types": output_types,
            "workflow_output_types": workflow_output_types,
            "doc_string": get_handler_doc_string(executor, input_type), # Note: handlers require unique input types
            "input_parameter_names": get_handler_parameter_names(executor, input_type),
        })

    return handler_info

In [75]:
get_function_executor_metadata(reverse_text_typed)

<class 'str'>


[{'handler_name': 'reverse_text_typed',
  'input_type': str,
  'output_types': [str],
  'workflow_output_types': [float],
  'doc_string': 'Reverse the input text and forward it to the next node.',
  'input_parameter_names': ['text']}]

In [71]:
# Note: I can get the FunctionExecutor class if it is from a function if that helps


# Note: class based executors can have multiple handlers
# Note: function based executors have a single handler, which is the function itself
# Note: class based executors require an id to be provided, function based executors use the function name as the id if not provided
# Note: handlers cannot have the same signature. If two handlers have union types overlap, handlers are registered in reverse order, so the most recently added handler that matches the input type will be used.
# Note: ^MAF does not have documentation for this behaviour, yet but I have tested to confirm it. Handlers are executed in the order they are returned from the executor's _handler_specs property, which is in reverse order of registration.

def get_executor_metadata(executor: Executor) -> dict:
    return {
        "id": executor.id,
        "type": executor.type,
        "handlers": executor._handler_specs,
    }

get_executor_metadata(upper_case_executor)
# get_executor_metadata(reverse_text)

{'id': 'upper_case_executor',
 'type': 'UpperCase',
 'handlers': [{'name': 'to_double_integer',
   'message_type': int,
   'output_types': [int],
   'workflow_output_types': [],
   'ctx_annotation': agent_framework._workflows._workflow_context.WorkflowContext[int, typing.Never]},
  {'name': 'to_lower_case',
   'message_type': str | int,
   'output_types': [str],
   'workflow_output_types': [],
   'ctx_annotation': agent_framework._workflows._workflow_context.WorkflowContext[str, typing.Never]},
  {'name': 'to_upper_case',
   'message_type': str,
   'output_types': [str],
   'workflow_output_types': [],
   'ctx_annotation': agent_framework._workflows._workflow_context.WorkflowContext[str, typing.Never]}]}

In [72]:
def get_handle_doc_string(executor: Executor, handler_name: str, message_type: str) -> str:

    handler = executor.
    print(executor._handlers)
    # for handler_spec in executor._handler_specs:
        
        # print(handler_spec.handler.__name__)

    """Get the docstring for a specific handler of an executor."""
    # for handler_spec in executor._handler_specs:
    #     if handler_spec.handler.__name__ == handler_name:
    #         return handler_spec.handler.__doc__
    # raise ValueError(f"Handler {handler_name} not found in executor {executor.id}")

get_handle_doc_string(upper_case_executor, "to_upper_case")

SyntaxError: invalid syntax (400995371.py, line 3)

In [ ]:
workflow = WorkflowBuilder(start_executor=upper_case_executor).add_edge(upper_case_executor, reverse_text).build()
result = await workflow.run("hello there") # Note: notebook runner already has an event loop running
print(result)

[WorkflowEvent(type='executor_invoked', executor_id='upper_case_executor', data='hello there'), WorkflowEvent(type='executor_completed', executor_id='upper_case_executor', data=['hel']), WorkflowEvent(type='superstep_started', iteration=1), WorkflowEvent(type='executor_invoked', executor_id='reverse_text', data='hel'), WorkflowEvent(type='executor_completed', executor_id='reverse_text', data=['leh']), WorkflowEvent(type='superstep_completed', iteration=1), WorkflowEvent(type='superstep_started', iteration=2), WorkflowEvent(type='superstep_completed', iteration=2)]


In [ ]:
[a for a in dir(reverse_text) if not a.startswith('__')]

['_create_context_for_handler',
 '_discover_handlers',
 '_discover_response_handlers',
 '_find_handler',
 '_find_response_handler',
 '_handler_specs',
 '_handlers',
 '_has_context',
 '_is_async',
 '_original_func',
 '_register_instance_handler',
 '_response_handler_specs',
 '_response_handlers',
 'can_handle',
 'clone',
 'execute',
 'from_dict',
 'from_json',
 'id',
 'input_types',
 'is_request_response_capable',
 'is_request_supported',
 'on_checkpoint_restore',
 'on_checkpoint_save',
 'output_types',
 'to_dict',
 'to_json',
 'type',
 'type_',
 'workflow_output_types']